# ArSL Word Training — Kaggle T4 Edition

**Upload these 3 files as a Kaggle dataset before running:**

| File | What it is |
|------|------------|
| `arsl_word_sequences_v2.npz` | Pre-extracted sequences (4023 × 48 × 258) |
| `KARSL-502_Labels.txt` | Arabic / English label names |
| `arsl_v2_classes.csv` | Class index mapping |

**Then:**
1. Add that dataset as input to this notebook
2. Update `DATASET_SLUG` in Cell 2 to match your dataset name
3. Set accelerator to **GPU T4 x1**
4. Run all cells

No MediaPipe, no video extraction — starts training in under 30 seconds.

### Architecture (T4 optimised — 16 GB VRAM):
```
Input (64, 48, 258)
  TimeDistributed Dense 384 → 256  [per-frame spatial encoder]
  BiLSTM(256) + BN + SpatialDropout
  BiLSTM(192) + BN + SpatialDropout
  LSTM(128, return_seq=True) + BN
  MultiHeadAttention(8 heads, key=64) + residual + LayerNorm
  GlobalAveragePooling1D
  Dense(512) + BN + Dropout
  Dense(256) + Dropout
  Softmax(num_classes)
```
~3M parameters · Mixed precision float16 · cuDNN-accelerated


In [ ]:
# ============================================================
# Cell 1 : Imports
# ============================================================
import os, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, classification_report
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, LSTM, Bidirectional, Dense, Dropout,
    BatchNormalization, TimeDistributed,
    MultiHeadAttention, GlobalAveragePooling1D,
    LayerNormalization
)
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import mixed_precision

warnings.filterwarnings('ignore')
print(f'TensorFlow : {tf.__version__}')
print(f'NumPy      : {np.__version__}')
print('Imports OK')


In [ ]:
# ============================================================
# Cell 2 : Configuration
# ← ONLY change DATASET_SLUG to match your Kaggle dataset name
# ============================================================

DATASET_SLUG = 'your-dataset-name'   # ← e.g. 'arsl-karsl-sequences'

INPUT_DIR  = Path(f'/kaggle/input/{DATASET_SLUG}')
OUTPUT_DIR = Path('/kaggle/working')

NPZ_PATH    = INPUT_DIR / 'arsl_word_sequences_v2.npz'
LABELS_FILE = INPUT_DIR / 'KARSL-502_Labels.txt'
CLASSES_CSV = INPUT_DIR / 'arsl_v2_classes.csv'

# ── Feature layout (must match what was extracted locally) ──
POSE_FEATURES = 33 * 4    # 132
HAND_FEATURES = 21 * 3    #  63 per hand
NUM_FEATURES  = POSE_FEATURES + HAND_FEATURES * 2   # 258
SEQUENCE_LENGTH = 48

# ── Hyperparameters (T4 / 16 GB VRAM) ──────────────────────
BATCH_SIZE      = 64
EPOCHS          = 200
LEARNING_RATE   = 5e-4
LSTM_UNITS_1    = 256
LSTM_UNITS_2    = 192
LSTM_UNITS_3    = 128
SPATIAL_ENC_1   = 384
SPATIAL_ENC_2   = 256
DENSE_UNITS     = 512
DROPOUT_RATE    = 0.4
LABEL_SMOOTH    = 0.1
GRAD_CLIP_NORM  = 1.0
TEST_SIZE       = 0.4
MHA_HEADS       = 8
MHA_KEY_DIM     = 64

# ── Verify input files ───────────────────────────────────────
print('=' * 55)
print('KAGGLE CONFIGURATION')
print('=' * 55)
for name, p in [('NPZ sequences', NPZ_PATH),
                ('Labels file',   LABELS_FILE),
                ('Classes CSV',   CLASSES_CSV)]:
    status = 'OK' if p.exists() else 'NOT FOUND'
    print(f'  [{status}] {name}: {p}')

if not NPZ_PATH.exists():
    raise FileNotFoundError(
        f'NPZ not found at {NPZ_PATH}\n'
        f'Make sure you added the dataset and DATASET_SLUG is correct.'
    )

print(f'\n  Features / frame  : {NUM_FEATURES}')
print(f'  Sequence length   : {SEQUENCE_LENGTH}')
print(f'  Batch size        : {BATCH_SIZE}')
print(f'  Max epochs        : {EPOCHS}')
print(f'  Spatial encoder   : {SPATIAL_ENC_1} -> {SPATIAL_ENC_2}')
print(f'  LSTM units        : {LSTM_UNITS_1} -> {LSTM_UNITS_2} -> {LSTM_UNITS_3}')
print(f'  MHA heads         : {MHA_HEADS} x key_dim {MHA_KEY_DIM}')
print(f'  Dense units       : {DENSE_UNITS}')


In [ ]:
# ============================================================
# Cell 3 : GPU Setup + Mixed Precision
# ============================================================
gpus = tf.config.list_physical_devices('GPU')
if not gpus:
    raise RuntimeError('No GPU found. Set accelerator to GPU T4 x1 in Kaggle settings.')

for g in gpus:
    tf.config.experimental.set_memory_growth(g, True)

# Mixed precision: float16 compute, float32 variables
# Gives ~2x speedup on T4 with no accuracy loss
mixed_precision.set_global_policy('mixed_float16')

print(f'GPU   : {gpus[0].name}')
try:
    d = tf.config.experimental.get_device_details(gpus[0])
    print(f'Name  : {d.get("device_name", "unknown")}')
    print(f'CC    : {d.get("compute_capability", "unknown")}')
except Exception:
    pass
print(f'Policy: {mixed_precision.global_policy().name}')
print('GPU ready.')


In [ ]:
# ============================================================
# Cell 4 : Load Labels
# ============================================================
id_to_english = {}
id_to_arabic  = {}

if LABELS_FILE.exists():
    with open(str(LABELS_FILE), 'r', encoding='utf-8', errors='replace') as fh:
        for line in fh:
            line = line.strip()
            if not line or line.lower().startswith('signid'):
                continue
            parts = line.split('\t')
            if len(parts) >= 3:
                try:
                    sid = int(parts[0])
                    ar  = parts[1].strip()
                    en  = parts[2].strip()
                    mapped_id = sid + 1   # labels are 0-indexed, folders are 1-indexed
                    id_to_english[mapped_id] = en if en and en not in ('?','??','') else str(mapped_id)
                    id_to_arabic[mapped_id]  = ar if ar and ar not in ('?','??','') else en
                except Exception:
                    continue
    print(f'Labels loaded : {len(id_to_english)} entries')
else:
    print('Labels file not found — numeric IDs will be used.')

# Also load classes CSV if available
if CLASSES_CSV.exists():
    classes_df = pd.read_csv(CLASSES_CSV)
    print(f'Classes CSV   : {len(classes_df)} rows')
    print(classes_df.head(5).to_string())


In [ ]:
# ============================================================
# Cell 5 : Load NPZ Dataset
# ============================================================
print('Loading NPZ ...')
t0 = time.time()
_d = np.load(str(NPZ_PATH))
X, y = _d['X'], _d['y']
print(f'Loaded in {time.time()-t0:.1f}s')
print(f'X shape : {X.shape}   ({X.nbytes/1e6:.0f} MB)')
print(f'y shape : {y.shape}')
print(f'Classes : {len(np.unique(y))}')

# Validate shape matches config
assert X.shape[1] == SEQUENCE_LENGTH, f'Sequence length mismatch: got {X.shape[1]}, expected {SEQUENCE_LENGTH}'
assert X.shape[2] == NUM_FEATURES,    f'Feature count mismatch: got {X.shape[2]}, expected {NUM_FEATURES}'
print('Shape validation passed.')

# Quick dataset summary
unique_ids, counts = np.unique(y, return_counts=True)
word_names = [id_to_english.get(int(uid), str(uid)) for uid in unique_ids]
print(f'Min samples/class : {counts.min()} ({word_names[counts.argmin()]})')
print(f'Max samples/class : {counts.max()} ({word_names[counts.argmax()]})')
print(f'Mean / Median     : {counts.mean():.1f} / {np.median(counts):.1f}')


In [ ]:
# ============================================================
# Cell 6 : Data Exploration
# ============================================================
sort_idx      = np.argsort(counts)[::-1]
sorted_names  = [word_names[i] for i in sort_idx]
sorted_counts = counts[sort_idx]

fig, axes = plt.subplots(1, 2, figsize=(22, 5))
axes[0].bar(range(len(sorted_names)), sorted_counts, color='darkgreen', edgecolor='black', linewidth=0.3)
axes[0].set_xticks(range(len(sorted_names)))
axes[0].set_xticklabels(sorted_names, rotation=90, fontsize=4)
axes[0].set_title(f'Class Distribution — {len(unique_ids)} classes, {len(y)} total samples', fontsize=13)
axes[0].axhline(np.mean(sorted_counts), color='red',    linestyle='--', alpha=0.7, label=f'Mean {np.mean(sorted_counts):.0f}')
axes[0].axhline(np.median(sorted_counts), color='orange', linestyle=':', alpha=0.7, label=f'Median {np.median(sorted_counts):.0f}')
axes[0].legend()

axes[1].hist(sorted_counts, bins=20, color='darkgreen', edgecolor='black', alpha=0.85)
axes[1].set_xlabel('Samples per class')
axes[1].set_ylabel('Number of classes')
axes[1].set_title('Samples per Class Distribution')
axes[1].axvline(np.mean(sorted_counts), color='red', linestyle='--', label=f'Mean {np.mean(sorted_counts):.0f}')
axes[1].legend()

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'class_distribution.png'), dpi=150)
plt.show()
print('Saved: class_distribution.png')


In [ ]:
# ============================================================
# Cell 7 : Preprocessing & Split
# ============================================================
print('Preprocessing ...')

# StandardScaler
orig_shape = X.shape
X_flat     = X.reshape(-1, NUM_FEATURES)
scaler     = StandardScaler()
X_flat     = scaler.fit_transform(X_flat)
X_scaled   = X_flat.reshape(orig_shape).astype(np.float32)

np.savez_compressed(str(OUTPUT_DIR / 'arsl_kaggle_scaler.npz'),
                    mean=scaler.mean_.astype(np.float32),
                    scale=scaler.scale_.astype(np.float32))
print('Scaler saved.')

# Label encoding
encoder     = LabelEncoder()
y_encoded   = encoder.fit_transform(y)
num_classes = len(encoder.classes_)
y_onehot    = to_categorical(y_encoded, num_classes=num_classes)

# Save class map
out_classes = pd.DataFrame({
    'model_class_index': range(num_classes),
    'karsl_class_id'   : encoder.classes_.tolist(),
    'english'          : [id_to_english.get(int(c), str(c)) for c in encoder.classes_],
    'arabic'           : [id_to_arabic.get(int(c),  str(c)) for c in encoder.classes_],
})
out_classes.to_csv(str(OUTPUT_DIR / 'arsl_kaggle_classes.csv'), index=False)
print(f'Class map saved ({num_classes} classes).')

# 60/20/20 split
try:
    X_tr, X_tmp, y_tr, y_tmp = train_test_split(
        X_scaled, y_onehot, test_size=TEST_SIZE, random_state=42, stratify=y_encoded
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_tmp, y_tmp, test_size=0.5, random_state=42, stratify=np.argmax(y_tmp, 1)
    )
except ValueError:
    print('WARNING: Stratified split failed (some classes have 1 sample) — using random split.')
    X_tr, X_tmp, y_tr, y_tmp = train_test_split(X_scaled, y_onehot, test_size=TEST_SIZE, random_state=42)
    X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.5, random_state=42)

# Balanced class weights
train_ints    = np.argmax(y_tr, axis=1)
cw_arr        = compute_class_weight('balanced', classes=np.arange(num_classes), y=train_ints)
cw_arr        = np.clip(cw_arr, 0.5, 10.0)
class_weights = dict(enumerate(cw_arr))

print(f'Train : {X_tr.shape[0]}  ({X_tr.shape[0]/len(X_scaled)*100:.0f}%)')
print(f'Val   : {X_val.shape[0]}  ({X_val.shape[0]/len(X_scaled)*100:.0f}%)')
print(f'Test  : {X_test.shape[0]}  ({X_test.shape[0]/len(X_scaled)*100:.0f}%)')
print(f'Input shape: {X_tr.shape[1:]}')


In [ ]:
# ============================================================
# Cell 8 : Build & Train — T4 Full Architecture
# ============================================================
tf.keras.backend.clear_session()

# ── Augmentation ─────────────────────────────────────────────
POSE_F = POSE_FEATURES
HAND_F = HAND_FEATURES

def augment_sequence(x, y_label):
    x = x + tf.random.normal(tf.shape(x), mean=0.0, stddev=0.005)
    x = tf.roll(x, shift=tf.random.uniform([], -3, 4, dtype=tf.int32), axis=0)
    mask = tf.cast(tf.random.uniform([SEQUENCE_LENGTH, 1]) > 0.1, tf.float32)
    x = x * mask
    x = x * tf.random.uniform([], 0.9, 1.1)
    # Horizontal flip: swap left/right hand blocks
    def flip():
        pose = x[:, :POSE_F]
        lh   = x[:, POSE_F : POSE_F + HAND_F]
        rh   = x[:, POSE_F + HAND_F :]
        return tf.concat([pose, rh, lh], axis=-1)
    x = tf.cond(tf.random.uniform([]) > 0.5, flip, lambda: x)
    return x, y_label

# ── tf.data pipelines ────────────────────────────────────────
AUTOTUNE = tf.data.AUTOTUNE
train_ds = (
    tf.data.Dataset.from_tensor_slices((X_tr, y_tr))
    .shuffle(min(len(X_tr), 10000), seed=42, reshuffle_each_iteration=True)
    .map(augment_sequence, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)
val_ds  = tf.data.Dataset.from_tensor_slices((X_val,  y_val)).batch(BATCH_SIZE).prefetch(AUTOTUNE)
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(BATCH_SIZE).prefetch(AUTOTUNE)
print('tf.data pipelines ready.')

# ── Model ─────────────────────────────────────────────────────
inputs = Input(shape=(SEQUENCE_LENGTH, NUM_FEATURES), name='input')

# Per-frame spatial encoder
x = TimeDistributed(Dense(SPATIAL_ENC_1, activation='relu',
                           kernel_initializer='he_normal',
                           kernel_regularizer=tf.keras.regularizers.l2(1e-4)),
                    name='td_1')(inputs)
x = TimeDistributed(BatchNormalization(), name='td_bn_1')(x)
x = TimeDistributed(Dropout(0.2),         name='td_drop_1')(x)
x = TimeDistributed(Dense(SPATIAL_ENC_2, activation='relu',
                           kernel_initializer='he_normal',
                           kernel_regularizer=tf.keras.regularizers.l2(1e-4)),
                    name='td_2')(x)
x = TimeDistributed(BatchNormalization(), name='td_bn_2')(x)

# Bidirectional temporal stack
x = Bidirectional(LSTM(LSTM_UNITS_1, return_sequences=True,
                        kernel_regularizer=tf.keras.regularizers.l2(1e-4)),
                  name='bilstm_1')(x)
x = BatchNormalization(name='bn_1')(x)
x = tf.keras.layers.SpatialDropout1D(DROPOUT_RATE, name='sdrop_1')(x)

x = Bidirectional(LSTM(LSTM_UNITS_2, return_sequences=True,
                        kernel_regularizer=tf.keras.regularizers.l2(1e-4)),
                  name='bilstm_2')(x)
x = BatchNormalization(name='bn_2')(x)
x = tf.keras.layers.SpatialDropout1D(DROPOUT_RATE, name='sdrop_2')(x)

x = LSTM(LSTM_UNITS_3, return_sequences=True,
          kernel_regularizer=tf.keras.regularizers.l2(1e-4),
          name='lstm_3')(x)
x = BatchNormalization(name='bn_3')(x)

# Multi-Head Attention
attn = MultiHeadAttention(num_heads=MHA_HEADS, key_dim=MHA_KEY_DIM,
                           dropout=0.1, name='mha')(query=x, value=x, key=x)
x = LayerNormalization(name='ln_attn')(x + attn)
x = GlobalAveragePooling1D(name='gap')(x)

# Classifier head
x = Dense(DENSE_UNITS, activation='relu',
           kernel_initializer='he_normal',
           kernel_regularizer=tf.keras.regularizers.l2(1e-4),
           name='dense_1')(x)
x = BatchNormalization(name='bn_d1')(x)
x = Dropout(DROPOUT_RATE, name='drop_1')(x)
x = Dense(DENSE_UNITS // 2, activation='relu',
           kernel_initializer='he_normal',
           kernel_regularizer=tf.keras.regularizers.l2(1e-4),
           name='dense_2')(x)
x = Dropout(DROPOUT_RATE * 0.5, name='drop_2')(x)

# Output must be float32 even with mixed precision
outputs = Dense(num_classes, activation='softmax', dtype='float32', name='output')(x)

model = Model(inputs, outputs, name='ArSL_Kaggle_T4')

# ── Optimizer + Loss ──────────────────────────────────────────
total_steps       = (len(X_tr) // BATCH_SIZE) * EPOCHS
first_decay_steps = max(total_steps // 5, 1)
lr_schedule = tf.keras.optimizers.schedules.CosineDecayRestarts(
    initial_learning_rate=LEARNING_RATE,
    first_decay_steps=first_decay_steps,
    t_mul=2.0, m_mul=0.9, alpha=1e-7
)
optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule, clipnorm=GRAD_CLIP_NORM)
loss_fn   = tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTH)

model.compile(
    optimizer=optimizer,
    loss=loss_fn,
    metrics=['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=5, name='top5_acc')]
)

print('Model summary:')
model.summary()

# ── Callbacks ─────────────────────────────────────────────────
# Use .keras format — Keras 3 (TF 2.16+) on Kaggle cannot reload
# MultiHeadAttention from legacy .h5 due to call-signature changes.
MODEL_BEST  = str(OUTPUT_DIR / 'arsl_kaggle_best.keras')
MODEL_FINAL = str(OUTPUT_DIR / 'arsl_kaggle_final.keras')

callbacks = [
    ModelCheckpoint(MODEL_BEST, monitor='val_accuracy',
                    save_best_only=True, mode='max', verbose=1),
    EarlyStopping(monitor='val_loss', patience=25,
                  restore_best_weights=True, verbose=1),
    tf.keras.callbacks.TerminateOnNaN(),
]

# ── Train ──────────────────────────────────────────────────────
print(f'\nStarting training on GPU...')
print(f'  Batch size   : {BATCH_SIZE}')
print(f'  Max epochs   : {EPOCHS}')
print(f'  LR schedule  : Cosine Annealing with warm restarts')
print(f'  Precision    : {mixed_precision.global_policy().name}')
print(f'  Augmentation : noise + shift + frame-drop + scale + LH<->RH flip')
print(f'  Class weights: balanced, clipped [0.5, 10]')
t_start = time.time()

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    class_weight=class_weights,
    verbose=1
)

elapsed = time.time() - t_start
print(f'\nTraining complete in {elapsed:.0f}s ({elapsed/60:.1f} min)')
print(f'  Best val_accuracy : {max(history.history["val_accuracy"]):.4f}')
print(f'  Best val_top5_acc : {max(history.history["val_top5_acc"]):.4f}')

model.save(MODEL_FINAL, save_format='keras')
print(f'\nSaved: {MODEL_BEST}')
print(f'Saved: {MODEL_FINAL}')


In [ ]:
# ============================================================
# Cell 9 : Evaluation Dashboard
# ============================================================
print('Loading best checkpoint ...')
best_model = tf.keras.models.load_model(MODEL_BEST)

proba  = best_model.predict(test_ds, verbose=0)
y_pred = np.argmax(proba, axis=1)
y_true = np.argmax(y_test, axis=1)

top1 = (y_pred == y_true).mean()
top5 = sum(1 for i in range(len(y_true))
           if y_true[i] in np.argsort(proba[i])[-5:]) / len(y_true)

print(f'\nTest Results:')
print(f'  Top-1 Accuracy : {top1:.4f} ({top1*100:.2f}%)')
print(f'  Top-5 Accuracy : {top5:.4f} ({top5*100:.2f}%)')
print(f'  Test samples   : {len(y_true)}')
print(f'  Classes        : {num_classes}')

word_labels = [
    id_to_english.get(int(encoder.classes_[i]), str(encoder.classes_[i]))
    for i in range(num_classes)
]

# ── Training curves ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(22, 5))
for ax, key, val_key, title in [
    (axes[0], 'accuracy',  'val_accuracy',  'Accuracy'),
    (axes[1], 'loss',      'val_loss',      'Loss'),
    (axes[2], 'top5_acc',  'val_top5_acc',  'Top-5 Accuracy'),
]:
    ax.plot(history.history[key],     label='Train', linewidth=2, color='#2E7D32')
    ax.plot(history.history[val_key], label='Val',   linewidth=2, color='#FF9800')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.legend(); ax.grid(True, alpha=0.3)
    if 'acc' in key: ax.set_ylim([0, 1.05])
plt.suptitle(f'Top-1: {top1*100:.1f}%  Top-5: {top5*100:.1f}%', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

# ── Per-class F1 ────────────────────────────────────────────
report = classification_report(
    y_true, y_pred, target_names=word_labels, zero_division=0, output_dict=True
)
print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=word_labels, zero_division=0))

class_f1   = {k: v['f1-score'] for k, v in report.items() if k in word_labels}
sorted_f1  = sorted(class_f1.items(), key=lambda x: x[1], reverse=True)
if sorted_f1:
    f1_names, f1_vals = zip(*sorted_f1)
    fig, ax = plt.subplots(figsize=(24, 6))
    colors = ['#4CAF50' if v >= 0.7 else '#FF9800' if v >= 0.4 else '#F44336' for v in f1_vals]
    ax.bar(range(len(f1_names)), f1_vals, color=colors, edgecolor='black', linewidth=0.3)
    ax.set_xticks(range(len(f1_names)))
    ax.set_xticklabels(f1_names, rotation=90, fontsize=4)
    ax.axhline(np.mean(f1_vals), color='blue', linestyle='--',
               label=f'Mean F1: {np.mean(f1_vals):.3f}')
    ax.set_title(f'Per-Class F1 (green>=0.7, orange>=0.4, red<0.4)', fontsize=13)
    ax.set_ylim([0, 1.05]); ax.legend()
    plt.tight_layout()
    plt.savefig(str(OUTPUT_DIR / 'f1_scores.png'), dpi=150, bbox_inches='tight')
    plt.show()

# ── Confusion matrix ─────────────────────────────────────────
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(20, 18))
sns.heatmap(cm, annot=(num_classes <= 50), fmt='d' if num_classes <= 50 else '',
            cmap='Greens', xticklabels=word_labels, yticklabels=word_labels, ax=ax)
ax.set_title(f'Confusion Matrix — {num_classes} classes  Top-1: {top1*100:.1f}%', fontsize=14)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.xticks(rotation=90, fontsize=4); plt.yticks(fontsize=4)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

# ── Top-10 confused pairs ────────────────────────────────────
cm_nd = cm.copy(); np.fill_diagonal(cm_nd, 0)
pairs = sorted(
    [(word_labels[i], word_labels[j], cm_nd[i, j])
     for i in range(num_classes) for j in range(num_classes) if cm_nd[i, j] > 0],
    key=lambda x: x[2], reverse=True
)[:10]
if pairs:
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.barh([f'{p[0]} -> {p[1]}' for p in pairs], [p[2] for p in pairs],
            color='#E91E63', edgecolor='darkred', alpha=0.85)
    ax.set_title('Top-10 Most Confused Pairs', fontsize=13, fontweight='bold')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig(str(OUTPUT_DIR / 'confused_pairs.png'), dpi=150, bbox_inches='tight')
    plt.show()

# ── Best / worst classes ─────────────────────────────────────
per_class_acc = {}
for i in range(num_classes):
    mask = y_true == i
    if mask.sum() > 0:
        per_class_acc[word_labels[i]] = (y_pred[mask] == i).mean()

sorted_acc = sorted(per_class_acc.items(), key=lambda x: x[1])
n_show = min(10, len(sorted_acc))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))
worst = sorted_acc[:n_show]
ax1.barh(range(len(worst)), [w[1]*100 for w in worst], color='#F44336', edgecolor='darkred', alpha=0.85)
ax1.set_yticks(range(len(worst))); ax1.set_yticklabels([w[0] for w in worst], fontsize=10)
ax1.set_title(f'Bottom {n_show} Classes', fontsize=13, color='#F44336', fontweight='bold')
ax1.set_xlim([0, 105])
for i, w in enumerate(worst): ax1.text(w[1]*100+1, i, f'{w[1]*100:.0f}%', va='center', fontsize=9)

best = sorted_acc[-n_show:][::-1]
ax2.barh(range(len(best)), [b[1]*100 for b in best], color='#4CAF50', edgecolor='darkgreen', alpha=0.85)
ax2.set_yticks(range(len(best))); ax2.set_yticklabels([b[0] for b in best], fontsize=10)
ax2.set_title(f'Top {n_show} Classes', fontsize=13, color='#4CAF50', fontweight='bold')
ax2.set_xlim([0, 105])
for i, b in enumerate(best): ax2.text(b[1]*100+1, i, f'{b[1]*100:.0f}%', va='center', fontsize=9)

plt.suptitle('Best vs Worst Performing Classes', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'best_worst_classes.png'), dpi=150, bbox_inches='tight')
plt.show()

print('\n' + '='*55)
print(f'DONE  Top-1: {top1*100:.2f}%   Top-5: {top5*100:.2f}%')
print('='*55)


## Output Files (all in `/kaggle/working/`)

| File | Description |
|------|-------------|
| `arsl_kaggle_best.keras` | Best checkpoint (highest val_accuracy) |
| `arsl_kaggle_final.keras` | Final model after all epochs |
| `arsl_kaggle_classes.csv` | model index ↔ class ID ↔ English ↔ Arabic |
| `arsl_kaggle_scaler.npz` | StandardScaler mean + scale for live inference |
| `training_curves.png` | Accuracy / loss / Top-5 plots |
| `confusion_matrix.png` | Full confusion matrix heatmap |
| `f1_scores.png` | Per-class F1 bar chart |
| `confused_pairs.png` | Top-10 most confused word pairs |
| `best_worst_classes.png` | Top-10 and bottom-10 class accuracy |
| `class_distribution.png` | Dataset class balance chart |

## Troubleshooting

| Issue | Fix |
|-------|-----|
| `NPZ not found` | Check `DATASET_SLUG` matches your Kaggle dataset name exactly |
| `No GPU found` | Kaggle settings → Accelerator → GPU T4 x1 |
| NaN loss | Set `LABEL_SMOOTH = 0` and `LEARNING_RATE = 1e-4` |
| OOM on T4 | Reduce `BATCH_SIZE` to 32 |
